# Stage 4. Postprocessing: Pareto front, priorities, weight map

**Question of the stage:** which strategy to choose under different priority settings of the regulator and how much this choice depends on weights and on the assessment of the damage of low-grade gasoline (c3). No new runs: the scenarios of stage 3 are used (`robust_scored.csv`, 9 finalists + 2 reference × 80 scenarios × 3 shocks) and, for context, the points of stage 2.

## Six profiles

As we discovered in stage 3, the best strategy mostly depends on two main things: the shock level and on the criterion of choice. The regulator may have different priorities: shortage, price, queues, or costs. We illustrate this with six profiles of weights for the four axes defined earlier in 02 during leveling. Here is the same table as in 02

| Axis | What it includes | Why |
|---|---|---|
| physical deficit | K0 | the main indicator of the crisis |
| price burden | K4, m.u. | K1 and K4 are both determined by price: in the screening their correlation across is 0.98–0.998. Two nearly identical axes add no information but increase the number of non-dominated points. K1 remains in the tables |
| queues | K5 above the no-shock background, h per car | the background (~2.7 h/car per year) exists even without a crisis; we compare what the shock and the measures add |
| cost of measures | K2 + c3 · K3, m.u., c3 = 0.25 | both quantities are the price of the measures for the budget and society. We reduce them to money via c3 (nominal 0.25 m.u./L — roughly the middle of the 0.05–0.5 range). Sensitivity to c3 is at stage 4, without runs |



Weights by axes (shortage / price / queues / costs), sum = 1:

| Profile | Weights |
|---|---|
| balanced | 0.25 / 0.25 / 0.25 / 0.25 |
| shortage_first (deficit above all) | 0.55 / 0.15 / 0.15 / 0.15 |
| price_first (consumer price) | 0.15 / 0.55 / 0.15 / 0.15 |
| budget_first (budget) | 0.15 / 0.15 / 0.15 / 0.55 |
| queues_first (social stability) | 0.15 / 0.15 / 0.55 / 0.15 |
| physical_security (deficit + queues) | 0.40 / 0.10 / 0.40 / 0.10 |

The main priority weighs 0.55, not 1.0: with a weight of 1 and zero for the others, a strategy that is catastrophic on other axes would win. With 0.55, the main priority weighs more than the other three combined (0.55 > 0.45), but they continue to constrain the choice. 



## How to calculate the score

The same as with stage 3: in each scenario (uncertainty point × shock × seed) each axis is normalized min–max over 11 strategies (0 = best and 1 = worst), the score = weighted sum of the normalized axes. The choice is based on the **90th percentile** of the score over 80 shock scenarios (a strategy that performs poorly in 10% of the worst scenarios is penalized); for comparison - by average. A robust choice for all shocks is the one with the lowest worst-case 90th percentile across three shocks. Only the finalists S1–S9 can be chosen; reference strategies participate in the normalization.

## 4.0. Environment

In [1]:
import json, itertools, math, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
import batch_runner as br

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
GRAY = "#b9b7b0"
# фиксированные цвета стратегий (одни и те же на всех рисунках этапа)
SCOL = {"S1": "#2a78d6", "S2": "#eb6834", "S3": "#1baf7a", "S4": "#eda100", "S5": "#8a5cd1",
        "S6": "#e0457b", "S7": "#13a3b8", "S8": "#7a8a1f", "S9": "#9c6b3c", "none": "#52514e", "L2_only": GRAY}
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.25, "figure.dpi": 110, "axes.prop_cycle": plt.cycler(color=PAL)})

OUT_DIR = Path("results")
PREP = json.loads((OUT_DIR / "prep_summary.json").read_text(encoding="utf-8"))
RBS  = json.loads((OUT_DIR / "robust_summary.json").read_text(encoding="utf-8"))
RS   = br.read_results(OUT_DIR / "robust_scored.csv")
LV   = br.read_results(OUT_DIR / "levels_scored.csv")

SHOCKS  = PREP["shock_levels"]
STRATS  = RBS["strategies"]
NAMES   = list(STRATS)
FIN     = [s for s in NAMES if s not in ("none", "L2_only")]
AXES    = ["K0", "K4", "K5_excess", "COST"]
AX_LAB  = {"K0": "shortage K0", "K4": "price burden K4", "K5_excess": "queues K5 above background", "COST": "policy cost K2 + c3*K3"}
SUB0    = PREP["interface_defaults"]["L4-subsidy"]       # субсидия, с которой считались прогоны (K2 ей пропорционален)
C3_NOM  = RBS["c3_nominal"]
C3_RANGE = PREP["c3_range"]
KEY     = ["u_id", "shock", "seed"]
PROFILES = {
    "balanced":          {"K0": 0.25, "K4": 0.25, "K5_excess": 0.25, "COST": 0.25},
    "shortage_first":    {"K0": 0.55, "K4": 0.15, "K5_excess": 0.15, "COST": 0.15},
    "price_first":       {"K0": 0.15, "K4": 0.55, "K5_excess": 0.15, "COST": 0.15},
    "budget_first":      {"K0": 0.15, "K4": 0.15, "K5_excess": 0.15, "COST": 0.55},
    "queues_first":      {"K0": 0.15, "K4": 0.15, "K5_excess": 0.55, "COST": 0.15},
    "physical_security": {"K0": 0.40, "K4": 0.10, "K5_excess": 0.40, "COST": 0.10},
}

# справочные K1 и K0 к 90-му дню относительно «без мер» в том же сценарии
none = RS.loc[RS.strategy == "none", KEY + ["K1", "K0@90"]].rename(columns={"K1": "K1_none", "K0@90": "K090_none"})
RS = RS.drop(columns=[c for c in ["K1_none", "K090_none"] if c in RS]).merge(none, on=KEY, how="left")
RS["K1_rel"] = RS.K1 / RS.K1_none
RS["K0_90_rel"] = RS["K0@90"] / RS.K090_none.clip(lower=1)
n_scen = RS.groupby(KEY).ngroups
print(f"stage-3 scenarios: {n_scen} ({n_scen // len(SHOCKS)} per shock), strategies: {len(NAMES)}")
pd.DataFrame([{"strategy": s, "point_id": v["point_id"], "role": v["role"]} for s, v in STRATS.items()])

stage-3 scenarios: 240 (80 per shock), strategies: 11


,strategy,point_id,role
0,S1,A31,best worst-case score across shocks
1,S2,A15,best at shock 0.2
2,S3,A53,best at shock 0.45
3,S4,A41,best at shock 0.7
4,S5,A09,closest to ideal at shock 0.45
5,S6,A23,closest to ideal at shock 0.7
6,S7,A30,minimum shortage (max supply)
7,S8,A59,"minimum overpayment, expensive"
8,S9,B36,best without price cap
9,none,none,reference


## 4.1. Pareto front of the finalists

A strategy is **Pareto-efficient** if no other strategy is better or equal to it on all four axes and strictly better on at least one. 


Here we calculate it in two ways:
* by **averages** over 80 shock scenarios - the "typical" front;
* by the **90th percentile** - the front for a cautious regulator (each axis is its value in the worst 10% of scenarios);
* and the **proportion of scenarios**, in which the strategy is Pareto-efficient among 11 - how often it is a reasonable choice in general.


In [ ]:
def nondominated(F):
    F = np.asarray(F, float)
    return np.array([not (np.all(F <= F[i], axis=1) & np.any(F < F[i], axis=1)).any() for i in range(len(F))])

REL = {"K0": "K0_rel", "K4": "K4_rel", "K5_excess": "K5_excess", "COST": "COST"}   # оси в читаемых единицах
rows = []
for sh in SHOCKS:
    d = RS[RS.shock == sh]
    mean = d.groupby("strategy")[list(REL.values()) + ["K1_rel", "K0_90_rel"]].mean().reindex(NAMES)
    p90 = d.groupby("strategy")[list(REL.values())].quantile(0.9).reindex(NAMES)
    nd_mean = nondominated(mean[list(REL.values())])
    nd_p90 = nondominated(p90)
    nd_rows = [pd.Series(nondominated(g[AXES]), index=g.strategy.to_numpy()) for _, g in d.groupby(KEY)]
    share = pd.concat(nd_rows, axis=1).mean(axis=1).reindex(NAMES)
    for s in NAMES:
        rows.append({"shock": sh, "strategy": s, **{f"{k}_mean": mean.loc[s, k] for k in mean.columns},
                     **{f"{REL[a]}_p90": p90.loc[s, REL[a]] for a in AXES},
                     "nd_mean": nd_mean[NAMES.index(s)], "nd_p90": nd_p90[NAMES.index(s)], "nd_scenario_share": share.loc[s]})
PARETO = pd.DataFrame(rows)
PARETO.to_csv(OUT_DIR / "stage4_pareto.csv", index=False, encoding="utf-8-sig")
for sh in SHOCKS:
    print(f"\n=== shock-depth = {sh} ===")
    display(PARETO[PARETO.shock == sh].drop(columns="shock").set_index("strategy").round(3))

In [ ]:
# проекции фронта по средним: подписи стратегий, недоминируемые - залитые маркеры
proj = [("K0_rel", "K4_rel"), ("K0_rel", "COST"), ("K5_excess", "K4_rel")]
lab = {"K0_rel": "K0, share of 'none'", "K4_rel": "K4, share of 'none'",
       "K5_excess": "queues above background, h/car", "COST": "policy cost, m.u."}
fig, axes = plt.subplots(len(proj), 3, figsize=(15, 12))
for j, sh in enumerate(SHOCKS):
    p = PARETO[PARETO.shock == sh]
    for i, (x, y) in enumerate(proj):
        ax = axes[i, j]
        for r in p.itertuples():
            xv, yv = getattr(r, x + "_mean"), getattr(r, y + "_mean")
            if x == "K0_rel":
                xv = max(xv, 1e-3)                    # лог-шкала: нулевой дефицит рисуем на 0.001
            if r.nd_mean:
                ax.scatter(xv, yv, s=60, color=SCOL[r.strategy], edgecolor="#0b0b0b", linewidth=1.0, zorder=3)
            else:
                ax.scatter(xv, yv, s=60, facecolor="white", edgecolor=SCOL[r.strategy], linewidth=2.0, zorder=3)
            ax.annotate(r.strategy, (xv, yv), fontsize=8, xytext=(4, 4), textcoords="offset points")
        if x == "K0_rel":
            ax.set_xscale("log")
        if x == "K5_excess":
            ax.set_xscale("symlog", linthresh=1)
        ax.set_xlabel(lab[x]); ax.set_ylabel(lab[y])
        if i == 0: ax.set_title(f"shock-depth = {sh}")
fig.suptitle("Finalists on the 4-axis front (means over 80 scenarios; filled = non-dominated, hollow = dominated)",
             x=0.01, ha="left")
plt.tight_layout(); plt.show()

### Рисунок для слайда: фронт этапа 2 и финалисты

Все 102 точки этапа 2 при номинальных параметрах (средние трёх seed): серые - доминируемые, голубые - недоминируемые по четырём осям, подписаны финалисты. Показывает, откуда взялись финалисты и как выглядит компромисс «дефицит ↔ переплата».

In [ ]:
pid2s = {v["point_id"]: s for s, v in STRATS.items() if s in FIN}
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
for ax, sh in zip(axes, SHOCKS):
    d = LV[LV.shock == sh]
    ax.scatter(d[~d.nd].K0_rel, d[~d.nd].K4_rel, s=12, color=GRAY, label="dominated")
    ax.scatter(d[d.nd].K0_rel, d[d.nd].K4_rel, s=16, color="#9ec5f0", label="non-dominated (4 axes)")
    for r in d[d.point_id.isin(pid2s)].itertuples():
        s = pid2s[r.point_id]
        ax.scatter(r.K0_rel, r.K4_rel, s=55, color=SCOL[s], edgecolor="#0b0b0b", zorder=3)
        ax.annotate(s, (r.K0_rel, r.K4_rel), fontsize=8, xytext=(4, 4), textcoords="offset points")
    a = d[d.point_id == "L2_only"]
    ax.scatter(a.K0_rel, a.K4_rel, s=55, marker="D", color="#0b0b0b", label="L2 only")
    ax.set_xscale("symlog", linthresh=0.01); ax.set_xlabel("K0, share of 'none'"); ax.set_ylabel("K4, share of 'none'")
    ax.set_title(f"shock-depth = {sh}")
axes[0].legend(frameon=False, fontsize=8)
fig.suptitle("Stage 2 design points (nominal parameters) and the selected finalists", x=0.01, ha="left")
plt.tight_layout(); plt.show()

## 4.2. Оценка при любых весах, c3 и субсидии

Функция пересчитывает затраты при заданных c3 и субсидии импорта (K2 пропорционален субсидии, K3 не зависит от прогонов), нормирует оси внутри сценариев и возвращает 90-й процентиль и среднее взвешенной оценки по каждой стратегии и шоку. Проверка: при равных весах и номинальных c3, субсидии результат совпадает с этапом 3.

In [ ]:
def build_norm(c3=C3_NOM, subsidy=SUB0):
    # тензор нормированных осей: по шоку -> массив (сценарии, стратегии, 4 оси)
    d = RS[KEY + ["strategy", "K0", "K4", "K5_excess", "K2", "K3_volume"]].copy()
    d["COST"] = d.K2 * subsidy / SUB0 + c3 * d.K3_volume
    out = {}
    for sh in SHOCKS:
        ds = d[d.shock == sh]
        arrs = []
        for a in AXES:
            m = ds.pivot_table(index=["u_id", "seed"], columns="strategy", values=a)[NAMES].to_numpy(float)
            lo, hi = m.min(axis=1, keepdims=True), m.max(axis=1, keepdims=True)
            arrs.append(np.where(hi > lo, (m - lo) / np.where(hi > lo, hi - lo, 1), 0.0))
        out[sh] = np.stack(arrs, axis=2)
    return out

def evaluate(weights, norm):
    w = np.array([weights[a] for a in AXES], float)
    res = {}
    for sh, N in norm.items():
        S = N @ w                                   # (сценарии, стратегии)
        res[sh] = pd.DataFrame({"p90": np.quantile(S, 0.9, axis=0), "mean": S.mean(axis=0)}, index=NAMES)
    return res

def choose(res, by="p90"):
    best = {sh: r.loc[FIN, by].idxmin() for sh, r in res.items()}
    worst = pd.concat({sh: r.loc[FIN, by] for sh, r in res.items()}, axis=1).max(axis=1)
    return best, worst.idxmin(), worst

NORM0 = build_norm()
res_eq = evaluate(PROFILES["balanced"], NORM0)
st3 = br.read_results(OUT_DIR / "robust_summary_table.csv").set_index(["shock", "strategy"]).score_p90
diff = max(abs(res_eq[sh].loc[s, "p90"] - st3.loc[(sh, s)]) for sh in SHOCKS for s in NAMES)
print(f"check vs stage 3 (equal weights, nominal c3 and subsidy): max |P90 difference| = {diff:.2e}")

## 4.3. Таблица «приоритеты → стратегия»

Для каждого профиля: лучшая стратегия при каждом шоке (по 90-му процентилю и по среднему) и одна устойчивая на все шоки. `margin` - насколько второй лучший хуже первого по 90-му процентилю (маленький margin = выбор почти равный, стоит смотреть и на второго).

In [ ]:
rows = []
for pname, w in PROFILES.items():
    res = evaluate(w, NORM0)
    best_p, rob_p, worst_p = choose(res, "p90")
    best_m, rob_m, _ = choose(res, "mean")
    r = {"profile": pname}
    for sh in SHOCKS:
        v = res[sh].loc[FIN, "p90"].sort_values()
        r[f"best@{sh}"] = best_p[sh]
        r[f"margin@{sh}"] = v.iloc[1] - v.iloc[0]
        r[f"best_by_mean@{sh}"] = best_m[sh]
    ws = worst_p.sort_values()
    r.update({"robust_all_shocks": rob_p, "robust_worst_p90": ws.iloc[0], "robust_runner_up": ws.index[1],
              "robust_margin": ws.iloc[1] - ws.iloc[0], "robust_by_mean": rob_m})
    rows.append(r)
PRIO = pd.DataFrame(rows).set_index("profile")
PRIO.to_csv(OUT_DIR / "stage4_priority_table.csv", encoding="utf-8-sig")
print("Priority profile -> recommended strategy (selection by P90 of the weighted score):")
display(PRIO[[f"best@{sh}" for sh in SHOCKS] + ["robust_all_shocks", "robust_runner_up", "robust_margin"]])
print("Margins and selection by mean (for comparison):")
display(PRIO[[f"margin@{sh}" for sh in SHOCKS] + [f"best_by_mean@{sh}" for sh in SHOCKS] + ["robust_by_mean"]].round(3))

In [ ]:
# что получает регулятор в каждой рекомендации: рычаги и средние KPI (справочно K1 и K0 к 90-му дню)
def levers(s):
    p = STRATS[s]["params"]
    return {"L3": p["L3-share-naphtha"] if p["L3-quality-relax?"] else 0.0,
            "L4": p["L4-import-limit"] if p["L4-import?"] else 0.0,
            "L5_L": p["L5-liters-per-fillup"] if p["L5-fillup-limit?"] else "off",
            "L6_cap": p["L6-price-cap"] if p["L6-price-cap?"] else "off"}

rows = []
for pname in PROFILES:
    for sh in SHOCKS:
        s = PRIO.loc[pname, f"best@{sh}"]
        m = PARETO[(PARETO.shock == sh) & (PARETO.strategy == s)].iloc[0]
        rows.append({"profile": pname, "shock": sh, "strategy": s, **levers(s),
                     "K0_rel": m.K0_rel_mean, "K4_rel": m.K4_rel_mean, "K5_h_per_car": m.K5_excess_mean,
                     "COST": m.COST_mean, "K1_rel": m.K1_rel_mean, "K0_day90_rel": m.K0_90_rel_mean})
RECO = pd.DataFrame(rows)
RECO.to_csv(OUT_DIR / "stage4_recommendations.csv", index=False, encoding="utf-8-sig")
RECO.round(3)

## 4.4. Карта весов

Ось x - вес дефицита, ось y - вес цены (их сумма не больше 1), остаток делится поровну между очередями и затратами. Цвет клетки - лучшая стратегия по 90-му процентилю: три карты по шокам и одна для устойчивого выбора на все шоки. Звёздочки - шесть профилей из 4.3. Последняя панель - запас устойчивого выбора (насколько второй лучший хуже первого): светлые зоны - выбор почти равный.

In [ ]:
STEP = 0.05
grid = [(round(a, 2), round(b, 2)) for a in np.arange(0, 1 + 1e-9, STEP) for b in np.arange(0, 1 - a + 1e-9, STEP)]

def weight_map(norm):
    rows = []
    for a, b in grid:
        rest = (1 - a - b) / 2
        res = evaluate({"K0": a, "K4": b, "K5_excess": rest, "COST": rest}, norm)
        best, rob, worst = choose(res, "p90")
        ws = worst.sort_values()
        rows.append({"w_K0": a, "w_K4": b, **{f"best@{sh}": best[sh] for sh in SHOCKS},
                     "robust": rob, "robust_margin": ws.iloc[1] - ws.iloc[0]})
    return pd.DataFrame(rows)

WMAP = weight_map(NORM0)
WMAP.to_csv(OUT_DIR / "stage4_weight_map.csv", index=False, encoding="utf-8-sig")

prof_xy = {p: (w["K0"], w["K4"]) for p, w in PROFILES.items() if abs(w["K5_excess"] - w["COST"]) < 1e-9}
cols = [f"best@{sh}" for sh in SHOCKS] + ["robust"]
titles = [f"best at shock-depth = {sh}" for sh in SHOCKS] + ["robust choice for all shocks"]
fig, axes = plt.subplots(1, 5, figsize=(24, 4.8))
for ax, c, t in zip(axes[:4], cols, titles):
    for s in FIN:
        d = WMAP[WMAP[c] == s]
        ax.scatter(d.w_K0, d.w_K4, s=95, marker="s", color=SCOL[s], linewidth=0)
        if len(d) >= 3:
            # подпись - в клетке региона, ближайшей к его центру (не попадает в чужой регион)
            cx, cy = d.w_K0.mean(), d.w_K4.mean()
            k = ((d.w_K0 - cx) ** 2 + (d.w_K4 - cy) ** 2).idxmin()
            ax.annotate(s, (d.loc[k, "w_K0"], d.loc[k, "w_K4"]), fontsize=8, ha="center", va="center", color="#0b0b0b",
                        fontweight="bold", bbox={"boxstyle": "round,pad=0.15", "fc": "white", "ec": "none", "alpha": 0.85})
    for p, (x, y) in prof_xy.items():
        ax.scatter(x, y, marker="*", s=120, color="#0b0b0b")
    ax.set_xlim(-0.04, 1.04); ax.set_ylim(-0.04, 1.04); ax.set_aspect("equal"); ax.grid(False)
    ax.set_xlabel("weight on shortage K0"); ax.set_ylabel("weight on price burden K4"); ax.set_title(t, fontsize=10)
sc = axes[4].scatter(WMAP.w_K0, WMAP.w_K4, c=WMAP.robust_margin, cmap="Blues", s=95, marker="s", linewidth=0)
axes[4].set_aspect("equal"); axes[4].grid(False); axes[4].set_title("robust choice: margin over runner-up", fontsize=10)
axes[4].set_xlabel("weight on shortage K0"); fig.colorbar(sc, ax=axes[4], shrink=0.7)
handles = [Patch(color=SCOL[s], label=f"{s} ({STRATS[s]['point_id']})") for s in FIN if (WMAP[cols] == s).any().any()]
fig.legend(handles=handles, loc="lower center", ncol=len(handles), frameon=False, fontsize=8)
fig.suptitle("Weight map (remaining weight split equally between queues and policy cost; stars = profiles with equal "
             "queue/cost weights)", x=0.01, ha="left")
plt.tight_layout(rect=(0, 0.06, 1, 0.94)); plt.show()

print("Share of the weight simplex where each strategy is the robust choice:")
display(WMAP.robust.value_counts(normalize=True).rename("share").to_frame().round(3))

## 4.5. Чувствительность к c3 и к субсидии импорта

c3 (ущерб от литра низкосортного бензина) и субсидия импорта не влияют на ход симуляции, поэтому их можно менять без прогонов. Вопрос: меняется ли рекомендация, если низкосортный бензин наносит больший или меньший ущерб, или импорт дороже. Особенно важно для стратегий с заметным L3 (S4, S7), где затраты сильно зависят от c3. Таблица: устойчивый выбор каждого профиля при разных c3 (субсидия номинальная); тепловая карта: устойчивый выбор сбалансированного профиля по сетке c3 × субсидия.

In [ ]:
C3_GRID = np.round(np.linspace(C3_RANGE[0], C3_RANGE[1], 10), 3)
SUB_GRID = [0.1, 0.2, SUB0, 0.35, 0.5]

rows = []
for c3 in C3_GRID:
    norm = build_norm(c3=c3)
    for pname, w in PROFILES.items():
        best, rob, _ = choose(evaluate(w, norm), "p90")
        rows.append({"c3": c3, "profile": pname, "robust": rob, **{f"best@{sh}": best[sh] for sh in SHOCKS}})
C3S = pd.DataFrame(rows)
C3S.to_csv(OUT_DIR / "stage4_c3_sensitivity.csv", index=False, encoding="utf-8-sig")
print("Robust choice by profile (rows) and c3 (columns), nominal subsidy:")
display(C3S.pivot(index="profile", columns="c3", values="robust"))

grid_cs = []
for sub in SUB_GRID:
    for c3 in C3_GRID:
        best, rob, _ = choose(evaluate(PROFILES["balanced"], build_norm(c3=c3, subsidy=sub)), "p90")
        grid_cs.append({"subsidy": sub, "c3": c3, "robust": rob})
GCS = pd.DataFrame(grid_cs)
GCS.to_csv(OUT_DIR / "stage4_c3_subsidy_balanced.csv", index=False, encoding="utf-8-sig")
M = GCS.pivot(index="subsidy", columns="c3", values="robust")
fig, ax = plt.subplots(figsize=(10, 2.8))
for i, sub in enumerate(M.index):
    for j, c3 in enumerate(M.columns):
        s = M.loc[sub, c3]
        ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, color=SCOL[s]))
        ax.text(j, i, s, ha="center", va="center", fontsize=8, color="white", fontweight="bold")
ax.set_xlim(-0.5, len(M.columns) - 0.5); ax.set_ylim(-0.5, len(M.index) - 0.5)
ax.set_xticks(range(len(M.columns)), M.columns); ax.set_yticks(range(len(M.index)), M.index); ax.grid(False)
ax.set_xlabel("c3, m.u. per litre of low-grade gasoline"); ax.set_ylabel("import subsidy, m.u./L")
ax.set_title("Balanced profile: robust choice under different cost assumptions", loc="left", fontsize=10)
plt.tight_layout(); plt.show()

## 4.6. Итог этапа

Сохраняется `stage4_summary.json`: рекомендации по профилям, доли карты весов, устойчивость к c3. Пришлите вывод этой ячейки, таблицы 4.3, рисунки 4.1 (проекции), 4.4 (карта весов) и 4.5.

In [ ]:
c3_stable = {p: C3S[C3S.profile == p].robust.nunique() == 1 for p in PROFILES}
SUMMARY = {
    "created": time.strftime("%Y-%m-%d %H:%M"), "model_sha256": PREP["model_sha256"],
    "profiles": PROFILES, "selection": "P90 of weighted score over 80 scenarios per shock",
    "priority_table": PRIO[[f"best@{sh}" for sh in SHOCKS] + ["robust_all_shocks", "robust_runner_up"]].to_dict(orient="index"),
    "weight_map_robust_share": WMAP.robust.value_counts(normalize=True).round(3).to_dict(),
    "robust_choice_stable_over_c3": c3_stable,
    "pareto_nondominated_mean": {str(sh): PARETO[(PARETO.shock == sh) & PARETO.nd_mean].strategy.tolist() for sh in SHOCKS},
}
(OUT_DIR / "stage4_summary.json").write_text(json.dumps(SUMMARY, indent=1, ensure_ascii=False, default=float), encoding="utf-8")
print(json.dumps({k: SUMMARY[k] for k in ["priority_table", "weight_map_robust_share", "robust_choice_stable_over_c3",
                                          "pareto_nondominated_mean"]}, indent=1, default=float))